# 🎬 Scalable Movie Recommendation System
### Hybrid ALS + Content-Based Filtering with PySpark · MovieLens 100K

This notebook walks through the entire pipeline end-to-end:

| Step | Description |
|------|-------------|
| 1 | Environment setup & data loading |
| 2 | Exploratory Data Analysis (EDA) |
| 3 | Feature engineering & preprocessing |
| 4 | ALS collaborative filtering |
| 5 | Content-based filtering (TF-IDF) |
| 6 | Hybrid ensemble & evaluation |
| 7 | Live recommendation demo |

---
## 0 · Setup

In [ ]:
# Install dependencies (run once)
# !pip install pyspark fastapi uvicorn scikit-learn

# Download dataset (run once)
# !bash ../data/download_data.sh

import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

%matplotlib inline
plt.rcParams.update({'figure.figsize': (10, 5), 'axes.spines.top': False, 'axes.spines.right': False})
print('✅ Imports OK')

---
## 1 · Start Spark & Load Data

In [ ]:
from preprocessing import get_spark, load_ratings, load_movies

spark = get_spark('NotebookDemo')
print(f'Spark version : {spark.version}')
print(f'Spark UI      : {spark.sparkContext.uiWebUrl}')

In [ ]:
ratings_df = load_ratings(spark)
movies_df  = load_movies(spark)

print(f'Ratings : {ratings_df.count():,} rows')
print(f'Movies  : {movies_df.count():,} rows')
ratings_df.show(5)
movies_df.select('movie_id','title','genre_string').show(5, truncate=False)

---
## 2 · Exploratory Data Analysis

In [ ]:
# Rating distribution
rating_counts = (
    ratings_df.groupBy('rating')
    .count()
    .orderBy('rating')
    .toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(rating_counts['rating'], rating_counts['count'], color='steelblue', width=0.6)
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Ratings per user (activity distribution)
user_activity = (
    ratings_df.groupBy('user_id').count()
    .toPandas()['count']
)
axes[1].hist(user_activity, bins=40, color='coral', edgecolor='white')
axes[1].set_title('Ratings per User (Activity Distribution)')
axes[1].set_xlabel('Number of ratings')
axes[1].set_ylabel('Number of users')

plt.tight_layout()
plt.show()

print(f'Average ratings per user : {user_activity.mean():.1f}')
print(f'Median  ratings per user : {user_activity.median():.0f}')
print(f'Max     ratings per user : {user_activity.max():,}')

In [ ]:
# Top-20 most-rated movies
from pyspark.sql import functions as F

top_movies = (
    ratings_df
    .groupBy('movie_id')
    .agg(F.count('rating').alias('n_ratings'),
         F.mean('rating').alias('avg_rating'))
    .join(movies_df.select('movie_id','title'), 'movie_id')
    .orderBy(F.desc('n_ratings'))
    .limit(20)
    .toPandas()
)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_movies['title'][::-1], top_movies['n_ratings'][::-1], color='mediumseagreen')
ax.set_title('Top 20 Most-Rated Movies', fontsize=14)
ax.set_xlabel('Number of ratings')
plt.tight_layout()
plt.show()

In [ ]:
# Genre popularity
from preprocessing import GENRES

genre_counts = {}
for g in GENRES:
    if g == 'unknown': continue
    count = movies_df.filter(F.array_contains('genres', g)).count()
    genre_counts[g] = count

genre_df = pd.DataFrame.from_dict(genre_counts, orient='index', columns=['count'])
genre_df = genre_df.sort_values('count', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(genre_df.index, genre_df['count'], color='mediumpurple')
ax.set_title('Movies per Genre', fontsize=14)
ax.set_xlabel('Number of movies')
plt.tight_layout()
plt.show()

---
## 3 · Preprocessing

In [ ]:
from preprocessing import normalize_ratings, train_test_split

ratings_norm = normalize_ratings(ratings_df)
train_df, test_df = train_test_split(ratings_norm, train_ratio=0.8)

print(f'Train size : {train_df.count():,}')
print(f'Test  size : {test_df.count():,}')
train_df.show(5)

---
## 4 · ALS Collaborative Filtering

In [ ]:
from als_model import ALSRecommender

als = ALSRecommender(rank=20, max_iter=15, reg_param=0.1)
als.fit(train_df)

In [ ]:
rmse = als.evaluate(test_df)
p, r = als.precision_recall_at_k(test_df, k=10)

print(f'\n── ALS Metrics ─────────────')
print(f'  RMSE         : {rmse:.4f}')
print(f'  Precision@10 : {p:.4f}')
print(f'  Recall@10    : {r:.4f}')

In [ ]:
# Hyperparameter sweep — rank vs RMSE
ranks = [5, 10, 15, 20, 30]
rmse_scores = []

for rk in ranks:
    m = ALSRecommender(rank=rk, max_iter=10)
    m.fit(train_df)
    rmse_scores.append(m.evaluate(test_df))

plt.plot(ranks, rmse_scores, 'o-', color='steelblue', linewidth=2)
plt.title('ALS Rank vs RMSE')
plt.xlabel('Rank (latent factors)')
plt.ylabel('RMSE')
plt.grid(alpha=0.3)
plt.show()

---
## 5 · Content-Based Filtering (TF-IDF)

In [ ]:
from content_based import ContentBasedRecommender

cb = ContentBasedRecommender(num_features=512)
cb.fit(movies_df)
print(f'Index size: {len(cb._movie_vectors):,} movies')

In [ ]:
# Movies similar to Toy Story
print('🎬 Movies similar to Toy Story (1995) [movie_id=1]:\n')
similar = cb.recommend_similar_movies(1, n=10)
for r in similar:
    print(f"  [{r['movie_id']:4d}] {r['title']:<45} score={r['content_score']:.4f}")

In [ ]:
# Content-based recommendations for user 1
print('👤 Content-based recommendations for user #1:\n')
user_recs = cb.recommend_for_user(1, ratings_df, n=10)
for r in user_recs:
    print(f"  [{r['movie_id']:4d}] {r['title']:<45} score={r['content_score']:.4f}")

---
## 6 · Hybrid Model & Evaluation

In [ ]:
from hybrid_recommender import HybridRecommender

hybrid = HybridRecommender(
    als_weight   = 0.6,
    als_rank     = 20,
    als_max_iter = 15,
    cb_num_features = 512,
)
hybrid.fit(train_df, ratings_df, movies_df)
metrics = hybrid.evaluate(test_df)

In [ ]:
# Visualise weight sensitivity
als_weights = [0.1, 0.2, 0.4, 0.6, 0.8, 0.9]
precision_scores, recall_scores = [], []

for w in als_weights:
    h = HybridRecommender(als_weight=w, als_rank=20, als_max_iter=10)
    h.fit(train_df, ratings_df, movies_df)
    m = h.evaluate(test_df)
    precision_scores.append(m['precision'])
    recall_scores.append(m['recall'])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(als_weights, precision_scores, 'o-', label='Precision@10', color='steelblue')
ax.plot(als_weights, recall_scores,    's-', label='Recall@10',    color='coral')
ax.axvline(0.6, linestyle='--', color='gray', label='Default (0.6)')
ax.set_title('ALS Weight vs Precision / Recall @10')
ax.set_xlabel('ALS Weight  (1 - weight = CB weight)')
ax.set_ylabel('Score')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7 · Live Recommendation Demo

In [ ]:
def show_user_recommendations(user_id, n=10):
    recs = hybrid.recommend(user_id, n=n)
    df = pd.DataFrame(recs)
    df.index = df.index + 1
    df.index.name = 'Rank'
    print(f'\n🎬 Top-{n} recommendations for user #{user_id}')
    print('=' * 80)
    display(df[['title', 'hybrid_score', 'als_score', 'content_score']]
            .style.background_gradient(subset=['hybrid_score'], cmap='Greens')
            .format({'hybrid_score': '{:.4f}', 'als_score': '{:.4f}', 'content_score': '{:.4f}'})
    )

for uid in [1, 42, 100]:
    show_user_recommendations(uid, n=10)

In [ ]:
# Similar movie demo
def show_similar_movies(movie_id, n=8):
    title = hybrid._movies_df.filter(F.col('movie_id') == movie_id).collect()[0]['title']
    recs  = hybrid.recommend_similar(movie_id, n=n)
    df    = pd.DataFrame(recs)
    df.index = df.index + 1
    print(f'\n🔍 Movies similar to "{title}"')
    display(df[['title','content_score']]
            .style.background_gradient(subset=['content_score'], cmap='Blues'))

for mid in [1, 50, 100]:
    show_similar_movies(mid)

In [ ]:
# Cleanup
spark.stop()
print('✅ Notebook complete · Spark session stopped')